# TopoSplit for HRRR Downward Shortwave Radiation Flux
Script takes HRRR DSWRF (global solar radiation on a flat plane) and partitions radiation into direct and diffuse components with the use of a clear-sky model and an empirically-derived regression model ([Arias et al., 2010](https://web.ujaen.es/investiga/tep220/pdf/2010_ruizarias_ecm.pdf)) (note: improved methods exist in [Arias and Gueymard, 2024](https://www.sciencedirect.com/science/article/pii/S0038092X24000574) but require the installation of [GISPLIT](https://github.com/jararias/gisplit/tree/main) and [caelus](https://github.com/jararias/caelus/tree/main) or more implimentation work).

Components are downscaled and topographic corrections (e.g., topogrpahic shading, sky view factor, etc.) are applied to each before joining again to generate dswrf_topo_in.

In [9]:
# libraries
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import math as m
import random

from insolation import insolf

import pygrib      # may not be needed if another library is used (xarray?)
from netCDF4 import Dataset

from osgeo import gdal, osr

In [10]:
# paths
UVU_OLSON_HOME = '/uufs/chpc.utah.edu/common/home/uvu-group1/olson/snow-data/'
HRRR_DATA = UVU_OLSON_HOME + 'HRRR/'
ERW_TOPO_FILE = UVU_OLSON_HOME + 'isnobal-data/ERW_topo.nc'

In [12]:
hrdir

<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x7f4c3c0b8630> >

In [17]:
# read in required HRRR datasets
time_utc = 16
hrrr_file = f"{HRRR_DATA}hrrr.20230301/hrrr.t{time_utc-6}z.wrfsfcf06.grib2"

hrdir = gdal.Open(hrrr_file)

# str(hrdir.timestep_for_band(10)) # ONLY FOR HRRR PARAMS
hrdir.GetRasterBand(10).GetMetadata()


# calculate time stamp for solar angles
# or use basename and convert to input for year, month, day

True

In [3]:
# read in topographic parameters

# uses ncdf4
with Dataset(ERW_TOPO_FILE) as dem:
    d = dem['dem'][:].astype(np.float64)
    vf = dem['sky_view_factor'][:].astype(np.float64)

# use gdal
topo = gdal.Open(ERW_TOPO_FILE, gdal.GA_ReadOnly)
topo1 = gdal.Open(topo.GetSubDatasets()[0][0])
dem = topo1.GetRasterBand(1).ReadAsArray() # convert elevation to type for topocalc
dem2 = dem.astype(np.double)
geotransform = topo1.GetGeoTransform()
x_resolution = geotransform[1]
x_resolution

# get long/lat from 



In [4]:
# run Clear-sky model

# function

def solar_geom(year, month, day, latitude, longitude, timezone):
    """
    Wrapper for insolf insolation functions
    
    Calculate solar geometries for single day
    Return:
      zenith
      azimuth 
      sv
    """
    # Generate Julian Day 
    jdrng = insolf.julian_day(year, 3, 1, np.arange(1, 25))

    # Step 2: Calculate Sun Position over the day
    sunv = insolf.sunvector(jdrng, latitude, longitude, timezone)
    azimuth, zenith = insolf.sunpos(sunv)

    # Calculate normal vectors for shading (sv)
    sv = insolf.normalvector(zenith, azimuth)

    return zenith, azimuth, sv


def hrrr_clearsky_dhi(hrdir, zenith_hr, jd, timezone, 
                        visibility=60, O3=0.02, alphag=0.2):
    """
    Wrapper for insolf insolation functions
    
    Calculate direct and diffuse insolation for a given hour and zenith angle based on DEM and other parameters.
    Returns:
        Idir_hr - HORIZONTAL
        Idif_hr - HORIZONTAL
    """

    # extract HRRR variables
    if hrdir.GetRasterBand(1).GetMetadata()['GRIB_COMMENT'] == 'Geopotential height [gpm]'
        dem = hrdir.GetRasterBand(1).ReadAsArray().astype(np.double)
    else:
        break
    if hrdir.GetRasterBand(2).GetMetadata()['GRIB_COMMENT'] == 'Temperature [C]'
        tempC = hrdir.GetRasterBand(2).ReadAsArray().astype(np.double)
        tempk = tempC + 273.15
    else:
        break
    if hrdir.GetRasterBand(3).GetMetadata()['GRIB_COMMENT'] == 'Relative humidity [%]'
        RH = hrdir.GetRasterBand(10).ReadAsArray().astype(np.double)
    else:
        break
 
    # Calculate insolationa dn return idir, idif arrays
    return insolf.insolation(zenith_hr, jd, dem, visibility, RH, tempK, O3, alphag)

### NEED to calculate all from HRRR and DEM!

# array for full day
zenith, azimuth, sv = solar_geom(year, month, day, latitude, longitude, timezone)

# iterate for each hour
idir, idif_hr = hrrr_clearsky_dhi(hrdir, zenith_hr, jd, timezone)

In [5]:
# calculate diffuse fraction


if hrdir.GetRasterBand(10).GetMetadata()['GRIB_COMMENT'] == 'Downward short-wave radiation flux [W/(m^2)]':
        dem = hrdir.GetRasterBand(10).ReadAsArray()
    else:
        break

In [6]:
# downsample and warp HRRR to domain

def gdal_output_bounds(topo):
    
    geo_transform = topo.GetGeoTransform()
    return [
        geo_transform[0],
        geo_transform[3] + geo_transform[5] * topo.RasterYSize,
        geo_transform[0] + geo_transform[1] * topo.RasterXSize,
        geo_transform[3]
    ]

def warp_hrrr(ERW_TOPO_FILE):


    topo = gdal.Open(ERW_TOPO_FILE, gdal.GA_ReadOnly)
    topo1 = gdal.Open(topo.GetSubDatasets()[0][0])
    spatial_info = osr.SpatialReference()
    spatial_info


    # Open the topo file (netCDF in your case)
    topo = gdal.Open(ERW_TOPO_FILE, gdal.GA_ReadOnly)
    
    # Open the first subdataset of topo file
    topo1 = gdal.Open(topo.GetSubDatasets()[0][0])
    
    # Create a spatial reference object
    spatial_info = osr.SpatialReference()
    
    # Set the spatial reference from the dataset's projection
    spatial_info.ImportFromWkt(topo1.GetProjection())
    
    # Now, you can get the EPSG code from the spatial reference
    epsg_code = spatial_info.GetAuthorityName(None) + ":" + spatial_info.GetAuthorityCode(None)
    
    # Print the EPSG code
    print(f"EPSG Code: {epsg_code}")
    
    # Define a temporary file for the warped output
    mem_hrrr_file = '/vsimem/grib_%i.tif' % random.getrandbits(32)
    
    # Create warp options
    options = gdal.WarpOptions(
        dstSRS=epsg_code,  # Use the EPSG code
        outputBoundsSRS=epsg_code,  # Set output bounds in the same SRS
        outputBounds=gdal_output_bounds(topo1),  # Replace with actual bounds
        xRes=topo1.GetGeoTransform()[1],  # Set pixel resolution in X
        yRes=topo1.GetGeoTransform()[1],  # Set pixel resolution in Y
        multithread=True  # Enable multithreading for performance
    )
    
    # Perform the warp (reprojection) of the raster data
    gdal.Warp(mem_hrrr_file, topo1, options=options)
    
    # Open the warped file
    hrrr_warp = gdal.Open(mem_hrrr_file, gdal.GA_ReadOnly)
    
    # Verify the result
    print(hrrr_warp.GetProjection())
    
    
    epsg_code = spatial_info.GetAuthorityName(None) + ":" + spatial_info.GetAuthorityCode(None)

    # temp file
    mem_hrrr_file = '/vsimem/grib_%i.tif' % random.getrandbits(32)
    
    # warp options for topo files
    options = gdal.WarpOptions(
                dstSRS=epsg_code,
                outputBoundsSRS=epsg_code,
                outputBounds= gdal_output_bounds(topo1),
                xRes=topo1.GetGeoTransform()[1],
                yRes=topo1.GetGeoTransform()[1],
                multithread=True,
                resampleAlg= "cubic" # "bilinear"
            )
    
    # warp file, original, options=topo1
    gdal.Warp(mem_hrrr_file, hrrr_file, options=options)
    
    ## 3.5 reopen (save)
    hrrr_warp = gdal.Open(mem_hrrr_file, gdal.GA_ReadOnly)


In [7]:
# apply topographic corrections and combine

In [ ]:
# visualize results